In [1]:
import pandas as pd
import sys                 # Permet de modifier les chemins de recherche de Python
from pathlib import Path   # Manipulation des chemins
PROJECT_ROOT = Path.cwd().parent   # C:\RetailVision
sys.path.insert(0, str(PROJECT_ROOT))
# Ajoute C:\RetailVision aux dossiers où Python cherche les modules

from src.etl.extract import extract_data
from src.etl.transform import (
    check_missing_value,
    check_duplicates_rows,
    check_duplicates_keys,
    check_data_types,
    data_quality_report,
    convert_datetime_columns,
    clean_orders
    

)
from src.etl.load import save_to_silver

In [2]:
datasets=extract_data()

In [3]:
primary_keys = {
    "olist_orders_dataset": "order_id",
    "olist_customers_dataset": "customer_id",
    "olist_products_dataset": "product_id",
    "olist_sellers_dataset": "seller_id",
    "olist_order_reviews_dataset": "review_id",
    "olist_order_payments_dataset": "order_id",
    "olist_order_items_dataset": "order_id",
    "olist_geolocation_dataset": None,
    "product_category_name_translation": "product_category_name"
}

In [4]:
# Pour Chaque Table
for table_name, df in datasets.items():

    pk = primary_keys[table_name]

    report = data_quality_report(df, pk)

    
    print(f"Table : {table_name}")
    print("=" * 40)
    print("")

    for key, value in report.items():
        print(f"{key}: {value}")
    
    print(end="\n"*2)  
  

Table : olist_customers_dataset

Rows: 99441
Columns: 99441
Missing Values Count: 0
Duplicate Rows: 0
Duplicate Keys: 0


Table : olist_geolocation_dataset

Rows: 1000163
Columns: 1000163
Missing Values Count: 0
Duplicate Rows: 261831
Duplicate Keys: None


Table : olist_orders_dataset

Rows: 99441
Columns: 99441
Missing Values Count: 4908
Duplicate Rows: 0
Duplicate Keys: 0


Table : olist_order_items_dataset

Rows: 112650
Columns: 112650
Missing Values Count: 0
Duplicate Rows: 0
Duplicate Keys: 13984


Table : olist_order_payments_dataset

Rows: 103886
Columns: 103886
Missing Values Count: 0
Duplicate Rows: 0
Duplicate Keys: 4446


Table : olist_order_reviews_dataset

Rows: 99224
Columns: 99224
Missing Values Count: 145903
Duplicate Rows: 0
Duplicate Keys: 814


Table : olist_products_dataset

Rows: 32951
Columns: 32951
Missing Values Count: 2448
Duplicate Rows: 0
Duplicate Keys: 0


Table : olist_sellers_dataset

Rows: 3095
Columns: 3095
Missing Values Count: 0
Duplicate Rows: 0
Dup

In [5]:
# Pour Toutes Les Tables En Meme Temps
reports = []
for table_name, df in datasets.items() :

    pk = primary_keys[table_name]
    report = data_quality_report(df,pk)
    report['Table'] = table_name
    reports.append(report)

summary = pd.DataFrame(reports)
print(summary)


      Rows  Columns  Missing Values Count  Duplicate Rows  Duplicate Keys  \
0    99441    99441                     0               0             0.0   
1  1000163  1000163                     0          261831             NaN   
2    99441    99441                  4908               0             0.0   
3   112650   112650                     0               0         13984.0   
4   103886   103886                     0               0          4446.0   
5    99224    99224                145903               0           814.0   
6    32951    32951                  2448               0             0.0   
7     3095     3095                     0               0             0.0   
8       71       71                     0               0             0.0   

                               Table  
0            olist_customers_dataset  
1          olist_geolocation_dataset  
2               olist_orders_dataset  
3          olist_order_items_dataset  
4       olist_order_payments_datas

In [6]:
list(datasets.keys())

['olist_customers_dataset',
 'olist_geolocation_dataset',
 'olist_orders_dataset',
 'olist_order_items_dataset',
 'olist_order_payments_dataset',
 'olist_order_reviews_dataset',
 'olist_products_dataset',
 'olist_sellers_dataset',
 'product_category_name_translation']

In [7]:
def Affichage_Dic(data):
    for key, value in data.items():
        print(f"- {key}: {value}")

In [8]:
customers  = datasets["olist_customers_dataset"]
reportCustomers=data_quality_report(customers ,primary_keys["olist_customers_dataset"])
print(" Customers Report : \n")
Affichage_Dic(reportCustomers)

 Customers Report : 

- Rows: 99441
- Columns: 99441
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [9]:
#Sauvegarder dans Silver
save_to_silver(customers,"olist_customers_dataset")

olist_customers_dataset : Saved


In [10]:
orders = datasets["olist_orders_dataset"]
orders.dtypes 

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [11]:
#Si on remarque probleme au niveau du type des dates
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
] #les collonnes qui contienent des Dates

orders = convert_datetime_columns(orders, date_columns)
orders.dtypes 


order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [12]:
reportOrders = data_quality_report(orders, primary_keys["olist_orders_dataset"])
print(" Orders Report : \n")
Affichage_Dic(reportOrders)

 Orders Report : 

- Rows: 99441
- Columns: 99441
- Missing Values Count: 4908
- Duplicate Rows: 0
- Duplicate Keys: 0


In [13]:
print("Missing Values pour Orders:")
check_missing_value(orders)

Missing Values pour Orders:


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [14]:
# nombre de commandes par statut pour les commandes où la date de livraison est manquante
orders[ orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [15]:
# Vérifier les lignes de dates manquantes
orders[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [16]:
orders[orders["order_delivered_carrier_date"].isnull()]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [17]:
orders[orders["order_approved_at"].isnull()]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [18]:
orders.groupby("order_status").count()


,order_id,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,,,
approved,2,2,2,2,0,0,2
canceled,625,625,625,484,75,6,625
created,5,5,5,0,0,0,5
delivered,96478,96478,96478,96464,96476,96470,96478
invoiced,314,314,314,314,0,0,314
processing,301,301,301,301,0,0,301
shipped,1107,1107,1107,1107,1107,0,1107
unavailable,609,609,609,609,0,0,609


In [19]:
# orders.groupby("order_status")["nom_colonne"].count() -> si tu veux une colonne specifique
orders.groupby("order_status")["order_delivered_customer_date"].count() 

order_status
approved           0
canceled           6
created            0
delivered      96470
invoiced           0
processing         0
shipped            0
unavailable        0
Name: order_delivered_customer_date, dtype: int64

In [20]:
for column in date_columns :
    print(f'Pour column : {column} :')
    print(orders[orders[column].isnull()].groupby("order_status")[column].size())
    print()

Pour column : order_purchase_timestamp :
Series([], Name: order_purchase_timestamp, dtype: int64)

Pour column : order_approved_at :
order_status
canceled     141
created        5
delivered     14
Name: order_approved_at, dtype: int64

Pour column : order_delivered_carrier_date :
order_status
approved         2
canceled       550
created          5
delivered        2
invoiced       314
processing     301
unavailable    609
Name: order_delivered_carrier_date, dtype: int64

Pour column : order_delivered_customer_date :
order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_delivered_customer_date, dtype: int64

Pour column : order_estimated_delivery_date :
Series([], Name: order_estimated_delivery_date, dtype: int64)



In [21]:
# DataFrame.groupby("colonne_de_regroupement")["colonne_à_analyser"]
#orders.groupby("order_status")["order_id"].size()

In [22]:
orders_clean = clean_orders(orders)
print(orders.shape)
print(orders_clean.shape)

(99441, 8)
(99418, 8)


In [29]:
save_to_silver(orders_clean,"olist_orders_dataset")

olist_orders_dataset : Saved
